# Week 58, Day 1: Decision trees: splitting by impurity

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w058/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Gini, entropy and the value of a split

Write gini(y) and entropy(y) for integer labels, and split_gain(y, left, impurity), where left is a boolean mask. Evaluate the split in the prediction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(581)

def gini(y):
    p = np.bincount(y) / len(y)
    return ...

def entropy(y):
    p = np.bincount(y) / len(y); p = p[p > 0]
    return ...

def split_gain(y, left, impurity):
    nl, nr = left.sum(), (~left).sum()
    return ...

y = np.array([0] * 50 + [1] * 50)
left = np.r_[np.ones(45, bool), np.zeros(50, bool), np.ones(5, bool)]
print(f"parent Gini {gini(y):.3f}; children {gini(y[left]):.3f} and {gini(y[~left]):.3f}; gain {split_gain(y, left, gini):.3f}")
print(f"parent entropy {entropy(y):.3f} bits; gain {split_gain(y, left, entropy):.3f} bits")

In [ ]:
import numpy as np
def test_gini():
    "gini returns 1 - sum p² for a vector of labels"
    assert abs(gini(np.array([0, 0, 1, 1])) - 0.5) < 1e-12 and gini(np.array([2, 2, 2])) == 0
def test_entropy():
    "entropy is in bits and equals 1 for a 50-50 node"
    assert abs(entropy(np.array([0, 1])) - 1) < 1e-12 and entropy(np.array([1, 1])) == 0
def test_gain():
    "split_gain is the parent impurity minus the weighted child impurities"
    y = np.array([0] * 50 + [1] * 50); left = np.r_[np.ones(45, bool), np.zeros(50, bool), np.ones(5, bool)]
    assert abs(split_gain(y, left, gini) - (0.5 - (50 * gini(y[left]) + 50 * gini(y[~left])) / 100)) < 1e-12

test_gini()
test_entropy()
test_gain()
print("All checks passed")

### Exercise: A tree for cell types

Complete best_split: for each feature, sort, compute the weighted Gini of every candidate split from cumulative counts, ignore positions where the value does not change, and keep the best. grow and predict_proba are provided; fit a depth-3 tree and report its rules and test accuracy.

In [ ]:
names = ["spike width", "firing rate", "burst index", "noise A", "noise B"]
def cells(n, g):
    y = g.integers(0, 3, n)
    width = np.array([0.25, 0.45, 0.65])[y] + 0.07 * g.normal(size=n)
    rate = np.array([25, 10, 4])[y] * np.exp(0.35 * g.normal(size=n))
    burst = np.array([0.05, 0.2, 0.4])[y] + 0.1 * g.normal(size=n)
    return np.c_[width, rate, burst, g.normal(size=n), g.random(n)], y

def gini_counts(c):
    n = c.sum(-1, keepdims=True)
    return 1 - np.sum((c / np.maximum(n, 1)) ** 2, axis=-1)

def best_split(X, y, K, feats=None):
    best = (None, None, np.inf); n = len(y)
    for j in (range(X.shape[1]) if feats is None else feats):
        o = np.argsort(X[:, j]); xs, ys = X[o, j], y[o]
        left = np.cumsum(np.eye(K)[ys], axis=0)[:-1]; right = left[-1] + np.eye(K)[ys[-1]] - left
        nl = np.arange(1, n); nr = n - nl
        imp = ...
        imp = np.where(xs[1:] > xs[:-1], imp, np.inf)
        i = int(np.argmin(imp))
        if imp[i] < best[2]:
            best = (j, (xs[i] + xs[i + 1]) / 2, imp[i])
    return best

def grow(X, y, K, depth, min_leaf=1):
    counts = np.bincount(y, minlength=K)
    if depth == 0 or len(y) < 2 * min_leaf or counts.max() == len(y):
        return ("leaf", counts)
    j, t, imp = best_split(X, y, K)
    if j is None or imp >= gini_counts(counts):
        return ("leaf", counts)
    m = X[:, j] <= t
    return ("node", j, t, grow(X[m], y[m], K, depth - 1, min_leaf), grow(X[~m], y[~m], K, depth - 1, min_leaf))

def predict_proba(tree, X, K):
    if tree[0] == "leaf":
        return np.tile(tree[1] / tree[1].sum(), (len(X), 1))
    _, j, t, L, R = tree; m = X[:, j] <= t; out = np.zeros((len(X), K))
    out[m], out[~m] = predict_proba(L, X[m], K), predict_proba(R, X[~m], K)
    return out

def rules(tree, depth=0):
    if tree[0] == "leaf":
        print("  " * depth + f"predict type {np.argmax(tree[1])} ({tree[1].tolist()})"); return
    _, j, t, L, R = tree
    print("  " * depth + f"if {names[j]} <= {t:.3g}:"); rules(L, depth + 1)
    print("  " * depth + "else:"); rules(R, depth + 1)

X, y = cells(300, r); X_te, y_te = cells(2000, r)
tree = grow(X, y, 3, depth=3)
acc_tree = np.mean(predict_proba(tree, X_te, 3).argmax(1) == y_te)
rules(tree)
print(f"test accuracy {acc_tree:.3f}")

In [ ]:
import numpy as np
def test_split():
    "best_split finds the feature and threshold that separate a toy set perfectly"
    Xs = np.array([[0.0, 5.0], [1.0, 4.0], [2.0, 1.0], [3.0, 0.0]]); ys = np.array([0, 0, 1, 1])
    j, t, imp = best_split(Xs, ys, 2)
    assert imp == 0 and ((j == 0 and 1 < t < 2) or (j == 1 and 1 < t < 4))
def test_tree():
    "a depth-3 tree reaches test accuracy above 0.85 and uses spike width or rate at the root"
    assert acc_tree > 0.85 and tree[1] in (0, 1)

test_split()
test_tree()
print("All checks passed")

### Exercise: A piecewise-constant tuning curve

Write best_reg_split(x, y) for one feature: sort, use cumulative sums to compute the total squared error of every split, and return the best threshold and its error. The recursive fitter is provided; compare training and test error for depths 1 to 6.

In [ ]:
def best_reg_split(x, y):
    o = np.argsort(x); xs, ys = x[o], y[o]; n = len(y)
    cs, cs2 = np.cumsum(ys)[:-1], np.cumsum(ys ** 2)[:-1]
    nl = np.arange(1, n); nr = n - nl
    sse_l = cs2 - cs ** 2 / nl
    sse_r = (np.sum(ys ** 2) - cs2) - (np.sum(ys) - cs) ** 2 / nr
    sse = ...
    i = int(np.argmin(sse))
    return (xs[i] + xs[i + 1]) / 2, sse[i]

def reg_tree(x, y, depth):
    if depth == 0 or len(y) < 4 or np.ptp(x) == 0:
        return ("leaf", y.mean())
    t, _ = best_reg_split(x, y); m = x <= t
    return ("node", t, reg_tree(x[m], y[m], depth - 1), reg_tree(x[~m], y[~m], depth - 1))

def reg_predict(tree, x):
    if tree[0] == "leaf":
        return np.full(len(x), tree[1])
    _, t, L, R = tree; out = np.empty(len(x)); m = x <= t
    out[m], out[~m] = reg_predict(L, x[m]), reg_predict(R, x[~m]); return out

curve = lambda c: 5 + 30 * c ** 2 / (c ** 2 + 0.15 ** 2)          # a contrast response function
c_tr, c_te = r.uniform(0, 1, 120), r.uniform(0, 1, 2000)
y_tr, y_te2 = curve(c_tr) + 3 * r.normal(size=120), curve(c_te) + 3 * r.normal(size=2000)
tr_err, te_err = [], []
for depth in range(1, 7):
    tr_ = reg_tree(c_tr, y_tr, depth)
    tr_err.append(np.mean((reg_predict(tr_, c_tr) - y_tr) ** 2)); te_err.append(np.mean((reg_predict(tr_, c_te) - y_te2) ** 2))
    print(f"depth {depth}: training MSE {tr_err[-1]:.1f}, test MSE {te_err[-1]:.1f} (noise variance 9)")

In [ ]:
import numpy as np
def test_split():
    "best_reg_split returns the threshold that minimises total squared error for a step function"
    xs = np.array([0.0, 1, 2, 3]); ys = np.array([1.0, 1, 5, 5])
    t, sse = best_reg_split(xs, ys)
    assert 1 < t < 2 and abs(sse) < 1e-12
def test_depth():
    "training error falls with depth, and depth 4 beats depth 1 on test data"
    assert tr_err[-1] < tr_err[0] and te_err[3] < te_err[0]

test_split()
test_depth()
print("All checks passed")